In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/72.0 MB 882.6 kB/s eta 0:01:21
   ---------------------------------------- 0.5/72.0 MB 882.6 kB/s eta 0:01:21
   ---------------------------------------- 0.8/72.0 MB 884.1 kB/s eta 0:01:21
    --------------------------------------- 1.0/72.0 MB 968.5 kB/s eta 0:01:14
    --------------------------------------- 1.3/72.0 MB 1.0 MB/s eta 0:01:09
   - -------------------------------------- 1.8/72.0 MB 1.2 MB/s eta 0:00:59
   - -------------------------------------- 2.1/72.0 MB 1.3 MB/s eta 0:00:56
   - -------------------------------------- 2.4/72.0 MB 1.2 MB/s eta 0:00:58
   - -------------------------------------- 2.4/72.0 MB 1.2 MB/s eta 0:00:58
   - -------------------------------------- 3.1/72.0 MB 1.3 MB/s eta 0:00:52
   - -------

In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import numpy as np

In [4]:
# Load dataset
df = pd.read_csv("Cleaned_GPS_Spoofing_Dataset.csv")

# Feature list
selected_features = ['TOW', 'PD', 'RX', 'TCD', 'CP', 'DO', 'PRN', 'CN0', 'PC', 'EC']

X = df[selected_features]
y = df['Output']

# Load scaler
scaler = joblib.load("scaler.joblib")
X_scaled = scaler.transform(X)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
# Calculate class weights
classes = np.unique(y_train)
class_counts = np.bincount(y_train)
total = len(y_train)

class_weights = {cls: total / class_counts[cls] for cls in classes}
class_weights

{0: 1.2833029598441525,
 1: 14.00342864979771,
 2: 11.541965749166337,
 3: 15.946587537091988}

In [7]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    tree_method="hist",
    objective="multi:softmax",
    num_class=4,
    scale_pos_weight=1,   # we use class weights instead
)

# Fit model with sample weights
weights = np.array([class_weights[c] for c in y_train])

xgb_model.fit(X_train, y_train, sample_weight=weights)

D:\Anaconda\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:51:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=4, ...)

In [8]:
y_pred = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9364190155328777

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97     79565
           1       0.76      0.91      0.83      7292
           2       0.68      0.91      0.78      8846
           3       0.92      0.99      0.95      6403

    accuracy                           0.94    102106
   macro avg       0.84      0.94      0.88    102106
weighted avg       0.95      0.94      0.94    102106


Confusion Matrix:
 [[74560  1270  3162   573]
 [   12  6654   626     0]
 [   16   799  8031     0]
 [   32     2     0  6369]]


In [9]:
joblib.dump(xgb_model, "xgb_model.joblib")
print("Model saved as xgb_model.joblib")

Model saved as xgb_model.joblib
